# Study 4, part 2: measuring what participants wrote

Each of the 117 participants rated 18 second works on transformativeness, essence change and
creativity, and wrote a justification after every rating: 6,318 justifications in all. This notebook
turns those justifications into numbers. It applies Study 1's eight word lists, the rules, without
changing them, and writes the tables the integration file (`Study4_03_Integration.Rmd`) needs for the
two registered text checks, the secondary refit and the exploratory description.

**The measurement is blind by construction.** The notebook reads six columns of the long data file:
the participant, the source, the kind of source, the second work, the question and the text. It never
reads the label, the source time or the rating, so nothing here can depend on them, and it has no blind
and unblinded modes. The integration file joins the label and the rating back on.

No person and no language model labels, codes or judges any response. A rule is a search pattern (a
written list of words and phrases, in any of their ordinary forms) that the computer applies in the
same way to every response. The one use of a pretrained model is the check in section 5, in which a
locally stored sentence-embedding model (all-mpnet-base-v2) turns each response into a list of numbers
so that a grouping by meaning can be compared with the topics.

Everything is computed by the shared module `shared/python/study_text.py`, which holds the measurement
half of the Study 1 text notebook. This notebook chooses the input, checks it, calls the module and
names the files.

Run it from this folder with the Python 3.12 kernel:

    HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 jupyter nbconvert --to notebook --execute --inplace \
      --ExecutePreprocessor.kernel_name=python312 Study4_02_Text.ipynb

---

# 0. Setup

The paths start from this notebook's folder. Three environment variables (settings the shell passes to
the program) can name other folders: `STUDY4_DIR` the Study 4 folder, `STUDY1_DIR` the Study 1 folder
whose deposited rules are compared below, and `STUDY_SHARED_PYTHON` the folder of the shared module.
Printed paths are relative to the pipeline folder, so the output names no account and no folder
particular to one machine.

The input is `study4_long_with_text.csv`, which `Study4_01_Quant.Rmd` writes to the Study 4 folder in
an unblinded run and to `blind_run/` in a blind run. The two files hold the same text and name the
same second work for every response. A blind run shuffles the label, the source time and the
modification column, and this notebook reads none of the three: it takes the second work from
`work_id`, which no shuffle touches. The notebook reads the unblinded file when it exists and the blind
file otherwise, and says which.

The registration names 20260903 as the seed of the text code. A seed is the number that starts a
random-number generator. The cell below makes it the seed of every random step in this notebook, all of
which sit in the topic model and its agreement check. The seed alone does not make a rerun give the
same numbers. The shared module also runs the k-means step of the agreement check, which splits the
responses into groups of similar meaning, on one processor thread (one strand of computation). On
several threads, k-means adds up its numbers in pieces, one per thread, and combines the pieces in the
order the threads finish, which changes from run to run; since rounding depends on that order, its
groups can differ. The seed and the single thread together make a rerun give the same numbers.

In [1]:
import datetime
import hashlib
import os
import platform
import sys
import warnings

# The Hugging Face libraries that load the embedding model are kept off the
# internet, so nothing is downloaded.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


def short_warning(message, category, filename, lineno, line=None):
    # A warning names the file it came from without the file's folder.
    return "%s: %s (%s, line %d)\n" % (category.__name__, message, os.path.basename(filename), lineno)


warnings.formatwarning = short_warning

import numpy as np
import pandas as pd

STUDY_DIR = os.path.abspath(os.environ.get("STUDY4_DIR", os.getcwd()))
STUDY1_DIR = os.path.abspath(os.environ.get("STUDY1_DIR", os.path.join(STUDY_DIR, os.pardir, "Study 1")))
SHARED_PYTHON = os.path.abspath(os.environ.get("STUDY_SHARED_PYTHON",
                                               os.path.join(STUDY_DIR, os.pardir, "shared", "python")))
sys.dont_write_bytecode = True
sys.path.insert(0, SHARED_PYTHON)
import study_text

# The registered seed of the text code. The ten refits of the topic model's
# stability rule use 20260903 to 20260912, and the final topic fit and the
# k-means grouping of the agreement check use 20260903.
TEXT_SEED = 20260903
TEXT_SEEDS = study_text.set_text_seed(TEXT_SEED)

show = study_text.display_path
INPUT_NAME = "study4_long_with_text.csv"
CANDIDATES = [os.path.join(STUDY_DIR, INPUT_NAME), os.path.join(STUDY_DIR, "blind_run", INPUT_NAME)]
INPUT_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert INPUT_PATH is not None, (
    "Cannot find %s or %s. Knit Study4_01_Quant.Rmd first." % tuple(show(p) for p in CANDIDATES))

NLP_DIR = os.path.join(STUDY_DIR, "nlp")
TABLES_DIR = os.path.join(STUDY_DIR, "tables")
os.makedirs(NLP_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# The previous run's manifest is deleted before this run writes anything, so
# a manifest on disk means the run that wrote it reached the last cell.
MANIFEST_PATH = os.path.join(TABLES_DIR, "study4_text_run_manifest.csv")
if os.path.exists(MANIFEST_PATH):
    os.remove(MANIFEST_PATH)

RUN_ID = study_text.new_run_id()


def file_sha256(path):
    # The SHA-256 checksum of a file's bytes: a 64-character string that
    # changes if any byte of the file changes.
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


INPUT_SHA256 = file_sha256(INPUT_PATH)
print("Input:          ", show(INPUT_PATH))
print("Input SHA-256:  ", INPUT_SHA256)
print("Writing data to:", show(NLP_DIR))
print("Writing tables: ", show(TABLES_DIR))
print("Run identifier: ", RUN_ID)
print("Seeds:          ", TEXT_SEEDS)

Input:           Study 4/study4_long_with_text.csv
Input SHA-256:   bd101904415b24bf57fbce03512f1fd002f9ff6cb277e87abd1d7872fd0a8117
Writing data to: Study 4/nlp
Writing tables:  Study 4/tables
Run identifier:  20261015T210236Z-1ebeef47
Seeds:           {'stability_refits_first': 20260903, 'stability_refits_last': 20260912, 'final_topic_fit': 20260903, 'kmeans_grouping': 20260903}


---

# 1. The blind slice

The blind slice is the only table the measurement sees. Only six columns are loaded from the long data
file, so the label, the source time, the modification column and the rating never enter this
notebook's memory.

A response is about one second work: a source at one modification level. The key column `stimulus_id`
therefore names the second work, such as `3NonFigure_Slight`, taken from `work_id`. The source, its
kind (with or without a human or humanoid figure) and the modification level travel beside it as
`stimulus_source`, `stimulus_type` and `stimulus_modification`. The modification level is read from
the work's own name, which no blind shuffle touches. The shared module accepts only columns named
`stimulus_...` beside the key and the text, and refuses any other column by name.

In [2]:
READ_COLUMNS = ["id", "source", "image_kind", "work_id", "dimension", "response_text"]
raw = pd.read_csv(INPUT_PATH, usecols=READ_COLUMNS, dtype=str, keep_default_na=False)
assert sorted(raw.columns) == sorted(READ_COLUMNS), "Unexpected columns were read."

# A work_id is a source and a modification level joined by an underscore.
parts = raw["work_id"].str.extract(r"^(\d(?:Figure|NonFigure))_(None|Slight|Dramatic)$")
assert parts.notna().all().all(), "Some work_id does not name a source and a modification level."
assert (parts[0] == raw["source"]).all(), "Some work_id names a source other than its row's."

blind = pd.DataFrame({
    "id": raw["id"],
    "stimulus_id": raw["work_id"],
    "stimulus_source": raw["source"],
    "stimulus_type": raw["image_kind"],
    "stimulus_modification": parts[1],
    "dimension": raw["dimension"],
    "response_text": raw["response_text"],
})
del raw, parts
study_text.check_blind(blind)

# The counts the registration states: 6,318 justifications from 117
# participants, none empty, 54 for each participant, three for each of the
# 18 second works a participant saw.
n_rows, n_people = len(blind), blind["id"].nunique()
print("Responses:", n_rows, "| participants:", n_people,
      "| second works:", blind["stimulus_id"].nunique(),
      "| sources:", blind["stimulus_source"].nunique())
print("Questions:", sorted(blind["dimension"].unique()))
assert n_rows == 6318 and n_people == 117
assert blind["stimulus_id"].nunique() == 30 and blind["stimulus_source"].nunique() == 10
assert (blind.groupby("id").size() == 54).all()
assert (blind.groupby("id")["stimulus_id"].nunique() == 18).all()
assert (blind.groupby(["id", "stimulus_id"])["dimension"].nunique() == 3).all()
assert (blind["response_text"].str.strip() != "").all(), "A justification is empty."
print("Columns:", list(blind.columns))

Responses: 6318 | participants: 117 | second works: 30 | sources: 10
Questions: ['creativity', 'essence_change', 'transformativeness']
Columns: ['id', 'stimulus_id', 'stimulus_source', 'stimulus_type', 'stimulus_modification', 'dimension', 'response_text']


---

# 2. The rules are Study 1's

The registration says Study 1's eight word lists run unchanged. Two checks show that they do.

First, the module's rules, written out as Study 1's notebook wrote them (with Study 1's name and its
2,220 responses), must equal Study 1's deposited `tables/text_rules.txt` character for character, and
their SHA-256 checksum must equal the one in Study 1's `tables/text_rules.sha256`.

Second, Study 4's own rules text may differ from Study 1's in two lines only: the title, which names
the study, and the line that gives the number of responses. The constructed sentences, written for the
check and taken from no corpus, must also be read as the rules say.

In [3]:
STUDY1_RULES = os.path.join(STUDY1_DIR, "tables", "text_rules.txt")
STUDY1_RULES_SHA = os.path.join(STUDY1_DIR, "tables", "text_rules.sha256")
STUDY1_N_RESPONSES = 2220

with open(STUDY1_RULES, encoding="utf-8") as handle:
    study1_rules_text = handle.read()
with open(STUDY1_RULES_SHA, encoding="utf-8") as handle:
    study1_sha = handle.read().split()[0]

as_study1 = study_text.format_rules(study_text.RULES, "Study 1", STUDY1_N_RESPONSES) + "\n"
assert as_study1 == study1_rules_text, "The module's rules differ from Study 1's deposited rules."
assert study_text.sha256_text(as_study1) == study1_sha == study_text.sha256_text(study1_rules_text), \
    "The SHA-256 of the rules differs from Study 1's deposited checksum."
print("Rules equal Study 1's tables/text_rules.txt: True")
print("SHA-256 equals Study 1's tables/text_rules.sha256:", study1_sha)

RULES_TEXT = study_text.rules_text(blind, "Study 4")
RULES_SHA256 = study_text.sha256_text(RULES_TEXT)
assert len(study1_rules_text.splitlines()) == len(RULES_TEXT.splitlines())
differing = [(a, b) for a, b in zip(study1_rules_text.splitlines(), RULES_TEXT.splitlines()) if a != b]
assert len(differing) == 2, "Study 4's rules differ from Study 1's in more than the two named lines."
print()
print("Lines in which Study 4's rules text differs from Study 1's:")
for a, b in differing:
    print("  Study 1:", a)
    print("  Study 4:", b)
print()
print("SHA-256 of Study 4's rules text:", RULES_SHA256)

probes = study_text.check_constructed_sentences()
print("Constructed sentences read as the rules say:", len(probes), "of", len(probes))

Rules equal Study 1's tables/text_rules.txt: True
SHA-256 equals Study 1's tables/text_rules.sha256: 6aa77dbc26c9086c5349a863c1cab808d7bd3758265fc365de2fb36c46a30239

Lines in which Study 4's rules text differs from Study 1's:
  Study 1: STUDY 1 TEXT MEASURES: THE RULES IN FULL
  Study 4: STUDY 4 TEXT MEASURES: THE RULES IN FULL
  Study 1: Every measure below is applied by computer to all 2,220 responses, identically,
  Study 4: Every measure below is applied by computer to all 6,318 responses, identically,

SHA-256 of Study 4's rules text: 1229bf2bf6e58a406a8d8104620ad2556d7d17dcb7e21f3c2bbc65c0abd2212e
Constructed sentences read as the rules say: 13 of 13


---

# 3. The measures

One row per response: the identifiers, the word count, the eight measures, the reading of machine-made
talk beside its flag, and the four polarity columns that say whether a detail, time, skill or concept
phrase was claimed or denied. The machine-made reading is "asserted" when a response states, plainly
or as an impression, that a machine made the work. The integration file uses that reading for the
secondary refit, which leaves out the Human-arm participants who wrote such a statement. The counts
below are over the whole corpus and are split by nothing.

In [4]:
measures = study_text.measures_frame(blind)
MEASURES_PATH = os.path.join(NLP_DIR, "study4_text_measures.csv")
study_text.write_csv_atomic(MEASURES_PATH, measures, RUN_ID)
print("Written:", show(MEASURES_PATH), "|", len(measures), "rows |", len(measures.columns), "columns")
print("Columns:", list(measures.columns))
print()
print("Responses each flag fires on, whole corpus:")
for flag in study_text.TRUE_FALSE_FLAGS:
    print("  %-30s %5d" % (flag, int(measures[flag].sum())))
for value in ["asserted", "negated"]:
    print("  %-30s %5d" % ("prior_exposure " + value, int((measures["prior_exposure"] == value).sum())))
print("  %-30s %5d" % ("machine_made_reading asserted",
                       int((measures["machine_made_reading"] == "asserted").sum())))

Written: Study 4/nlp/study4_text_measures.csv | 6318 rows | 20 columns
Columns: ['id', 'stimulus_id', 'stimulus_source', 'stimulus_type', 'stimulus_modification', 'dimension', 'n_words', 'says_detail', 'says_time', 'says_skill', 'says_concept', 'says_prompt', 'says_machine_made', 'machine_made_reading', 'prior_exposure', 'anchor_echo', 'detail_polarity', 'time_polarity', 'skill_polarity', 'concept_polarity']

Responses each flag fires on, whole corpus:
  says_detail                      319
  says_time                       1457
  says_skill                        50
  says_concept                     776
  says_prompt                      182
  says_machine_made                 87
  anchor_echo                      395
  prior_exposure asserted          137
  prior_exposure negated            13
  machine_made_reading asserted     82


---

# 4. The word tables for the two registered checks

Three tables carry the words of each response to the integration file. A content word is a word left
after the fixed list of function words, such as *the* and *of*, is removed. The word-count table has one
row per response and content word that appears in at least two responses. The vocabulary lists those
words with the number of responses each appears in, and the content-word table gives each response's
distinct content words. The first check predicts the rating and the label from the counts. The second
compares how much participants reuse their own words, using the content-word lists. The registration
fixes both checks for creativity, and the tables cover every question so that the exploratory
description can use them too.

In [5]:
WORDCOUNTS_PATH = os.path.join(NLP_DIR, "study4_text_wordcounts.csv")
VOCABULARY_PATH = os.path.join(NLP_DIR, "study4_text_vocabulary.csv")
TOKENS_PATH = os.path.join(NLP_DIR, "study4_text_tokens.csv")
wordcounts = study_text.wordcounts_frame(blind)
vocabulary = study_text.vocabulary_frame(blind)
tokens = study_text.tokens_frame(blind)
study_text.write_csv_atomic(WORDCOUNTS_PATH, wordcounts, RUN_ID)
study_text.write_csv_atomic(VOCABULARY_PATH, vocabulary, RUN_ID)
study_text.write_csv_atomic(TOKENS_PATH, tokens, RUN_ID)
for path, frame in [(WORDCOUNTS_PATH, wordcounts), (VOCABULARY_PATH, vocabulary), (TOKENS_PATH, tokens)]:
    print("Written:", show(path), "|", len(frame), "rows")

Written: Study 4/nlp/study4_text_wordcounts.csv | 35598 rows
Written: Study 4/nlp/study4_text_vocabulary.csv | 1536 rows
Written: Study 4/nlp/study4_text_tokens.csv | 6318 rows


---

# 5. The exploratory description

**Tone and concreteness.** Each response gets the mean valence (how pleasant a word is), arousal (how
excited it makes a reader feel), dominance (how much in control it makes a reader feel) and concreteness
of its content words, from published word ratings (Warriner, Kuperman & Brysbaert, 2013; Brysbaert,
Warriner & Kuperman, 2014). The words of the rating questions are removed first. A response with no
matched word gets an empty score.

**Topics.** A topic model finds groups of words that tend to occur in the same responses. The model is
non-negative matrix factorisation (it writes the table of responses by words as the product of two
tables with no negative entries, one giving each response's weight on each topic), fitted to each
question separately and given no labels, with the number of topics chosen by the stability rule in the
module.

**The agreement check.** The sentence-embedding model turns each response into 768 numbers, the
responses to each question are split into as many groups as that question has topics, and the split is
compared with each response's main topic by the adjusted Rand index (agreement between two groupings
beyond chance: 0 for chance, 1 for identical). The ten refits of the stability rule use the seeds
20260903 to 20260912, and the final topic fit and the grouping use 20260903. Nothing the embedding
model produces is kept beyond that index.

In [6]:
AFFECT_PATH = os.path.join(NLP_DIR, "study4_text_affect.csv")
affect = study_text.affect_frame(blind)
study_text.write_csv_atomic(AFFECT_PATH, affect, RUN_ID)
print("Written:", show(AFFECT_PATH), "|", len(affect), "rows")
print("Responses with no word in the valence norms:", int((affect["n_vad_matched"] == 0).sum()))
print("Responses with no word in the concreteness norms:", int((affect["n_conc_matched"] == 0).sum()))

Written: Study 4/nlp/study4_text_affect.csv | 6318 rows
Responses with no word in the valence norms: 523
Responses with no word in the concreteness norms: 414


In [7]:
TOPICS_PATH = os.path.join(NLP_DIR, "study4_text_topics.csv")
TOP_WORDS_PATH = os.path.join(TABLES_DIR, "study4_text_topics_top_words.csv")
STABILITY_PATH = os.path.join(TABLES_DIR, "study4_text_topics_stability.csv")
AGREEMENT_PATH = os.path.join(TABLES_DIR, "study4_text_topics_embedding_agreement.csv")

topics = study_text.topics_frame(blind)
top_words = study_text.topic_top_words_frame(blind)
stability = study_text.topic_stability_frame(blind)
fit = study_text.fit_topics(blind)
study_text.write_csv_atomic(TOPICS_PATH, topics, RUN_ID)
study_text.write_csv_atomic(TOP_WORDS_PATH, top_words, RUN_ID)
study_text.write_csv_atomic(STABILITY_PATH, stability, RUN_ID)
for dim, fitted in fit["by_dimension"].items():
    print("%-18s topics: %d | vocabulary: %d | refits stopped at the iteration limit: %d of %d | final fit settled: %s"
          % (dim, fitted["k"], fitted["n_vocab"], fitted["n_short"], fitted["n_refits"],
             fitted["final_converged"]))
print()
print(top_words[top_words["rank"] <= 5].groupby(["dimension", "topic"])["word"]
      .apply(", ".join).to_string())

agreement = study_text.embedding_agreement_frame(blind)
EMBEDDING_STATUS = agreement.attrs.get("embedding_status", "")
study_text.write_csv_atomic(AGREEMENT_PATH, agreement, RUN_ID)
print()
print("Embedding model:", study_text.EMBEDDING_MODEL_NAME, "|", EMBEDDING_STATUS)
print(agreement.to_string(index=False))

creativity         topics: 4 | vocabulary: 325 | refits stopped at the iteration limit: 0 of 80 | final fit settled: True
essence_change     topics: 8 | vocabulary: 330 | refits stopped at the iteration limit: 0 of 80 | final fit settled: True
transformativeness topics: 3 | vocabulary: 338 | refits stopped at the iteration limit: 0 of 80 | final fit settled: True

dimension           topic
creativity          1                 changed, different, style, colors, color
                    2                   first, second, looks, think, identical
                    3                        change, see, color, style, little
                    4                   difference, see, think, drawings, tell
essence_change      1            different, style, feeling, colors, completely
                    2                    first, second, compared, felt, unique
                    3                  essence, changes, think, similar, feels
                    4                      change, see


Embedding model: sentence-transformers/all-mpnet-base-v2 | loaded from the local copy
         dimension  k    ari
        creativity  4 0.1043
    essence_change  8 0.1125
transformativeness  3 0.0811


---

# 6. Export

The rules, their checksum and their grounding notes go to `tables/`, with a record of the software and a
run manifest. Every file name carries the `study4_` prefix, and the checksum line names
`study4_text_rules.txt`, so `shasum -a 256 -c study4_text_rules.sha256` run from `tables/` checks it.
Every data file is read back from disk and held to its agreed columns, none of which is the label, the
source time, the rating or the text. The manifest lists every output with its size and SHA-256, beside
the run identifier, the input's SHA-256 and the rules checksum, and it is written last.

In [8]:
RULES_PATH = os.path.join(TABLES_DIR, "study4_text_rules.txt")
RULES_SHA_PATH = os.path.join(TABLES_DIR, "study4_text_rules.sha256")
GROUNDING_PATH = os.path.join(TABLES_DIR, "study4_text_rules_grounding.txt")
ENVIRONMENT_PATH = os.path.join(TABLES_DIR, "study4_text_environment.csv")

study_text.write_text_atomic(RULES_PATH, RULES_TEXT, RUN_ID)
study_text.write_text_atomic(RULES_SHA_PATH, "%s  study4_text_rules.txt\n" % RULES_SHA256, RUN_ID)
study_text.write_text_atomic(GROUNDING_PATH, study_text.grounding_text(blind, "Study 4"), RUN_ID)
assert file_sha256(RULES_PATH) == RULES_SHA256


def package_version(module_name):
    try:
        module = __import__(module_name)
    except Exception:
        return "not installed"
    return str(getattr(module, "__version__", "installed, version not recorded"))


environment = pd.DataFrame([
    ("run_id", RUN_ID),
    ("python", platform.python_version()),
    ("platform", platform.platform()),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("scikit-learn", package_version("sklearn")),
    ("scipy", package_version("scipy")),
    ("sentence-transformers", package_version("sentence_transformers")),
    ("torch", package_version("torch")),
    ("embedding_model", "%s, %s" % (study_text.EMBEDDING_MODEL_NAME, EMBEDDING_STATUS)),
    ("seed_text_code", str(TEXT_SEED)),
    ("seed_stability_refits", "%d to %d" % (TEXT_SEEDS["stability_refits_first"],
                                            TEXT_SEEDS["stability_refits_last"])),
    ("seed_final_topic_fit", str(TEXT_SEEDS["final_topic_fit"])),
    ("seed_kmeans_grouping", str(TEXT_SEEDS["kmeans_grouping"])),
    ("kmeans_openmp_threads", str(study_text.KMEANS_OPENMP_THREADS)),
], columns=["item", "value"])
# The seeds in force at the end of the run are the ones set at its start.
assert study_text.text_seeds() == TEXT_SEEDS
study_text.write_csv_atomic(ENVIRONMENT_PATH, environment, RUN_ID)
print(environment.to_string(index=False))
print()

# Every data file, read back from disk and held to its agreed columns. No
# file may carry the label, the source time, the modification column, the
# rating or the text. Every file keyed by response, apart from the word-count
# table (which has no row for a response with no counted word), covers
# exactly the 6,318 responses.
FORBIDDEN = ["label", "source_time", "modification", "Type", "rating", "response_text",
             "condition", "arm", "age", "gender", "experience"]
DATA_CONTRACTS = [
    (MEASURES_PATH, study_text.export_columns(blind)),
    (WORDCOUNTS_PATH, study_text.WORDCOUNT_COLUMNS),
    (VOCABULARY_PATH, study_text.VOCABULARY_COLUMNS),
    (TOKENS_PATH, study_text.TOKENS_COLUMNS),
    (AFFECT_PATH, study_text.AFFECT_COLUMNS),
    (TOPICS_PATH, study_text.TOPICS_COLUMNS),
]
response_keys = set(blind[study_text.KEY].itertuples(index=False, name=None))
for path, columns in DATA_CONTRACTS:
    on_disk = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(on_disk.columns) == list(columns), "%s has the wrong columns." % show(path)
    assert not [c for c in FORBIDDEN if c in on_disk.columns], "%s carries a forbidden column." % show(path)
    study_text.check_output(on_disk, show(path))
    if path not in (WORDCOUNTS_PATH, VOCABULARY_PATH):
        keys = set(on_disk[study_text.KEY].itertuples(index=False, name=None))
        assert keys == response_keys, "%s does not cover exactly the 6,318 responses." % show(path)
    print("  %-38s %6d rows, %2d columns as agreed, no forbidden column"
          % (show(path), len(on_disk), len(columns)))

outputs = [MEASURES_PATH, WORDCOUNTS_PATH, VOCABULARY_PATH, TOKENS_PATH, AFFECT_PATH, TOPICS_PATH,
           TOP_WORDS_PATH, STABILITY_PATH, AGREEMENT_PATH, RULES_PATH, RULES_SHA_PATH,
           GROUNDING_PATH, ENVIRONMENT_PATH]
completed_utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
manifest = pd.DataFrame([dict(
    run_id=RUN_ID, completed_utc=completed_utc,
    input_file=os.path.relpath(INPUT_PATH, STUDY_DIR), input_sha256=INPUT_SHA256,
    rules_sha256=RULES_SHA256,
    file=os.path.relpath(path, STUDY_DIR), bytes=os.path.getsize(path), sha256=file_sha256(path),
) for path in outputs])
study_text.write_csv_atomic(MANIFEST_PATH, manifest, RUN_ID)
print()
print("Manifest:", show(MANIFEST_PATH), "|", len(manifest), "outputs listed, written last.")

                 item                                                               value
               run_id                                           20261015T210236Z-1ebeef47
               python                                                              3.12.3
             platform                                        macOS-26.6.2-arm64-arm-64bit
               pandas                                                               2.2.2
                numpy                                                              1.26.4
         scikit-learn                                                               1.5.2
                scipy                                                              1.13.0
sentence-transformers                                                               3.3.1
                torch                                                               2.5.1
      embedding_model sentence-transformers/all-mpnet-base-v2, loaded from the local copy
       see

---

## What this notebook does not write

Study 1's text notebook also writes four reports that this notebook does not: the matched and unmatched
responses for each rule and the list of candidate words, which are printed for a person to read, and the
distinctive-word tables and the corpus summary by arm, which need the rating or the condition. The
measurement here never reads either.